# Selene-Core Advanced: Extending the Build Planner

This notebook adds a custom artifact kind and step for a new input resource:
- Kind: a `.c` file that contains `int qmain`
- Step: compile that C file to a Helios object (`.o`)

Then we register those with a planner and let built-in steps finish the path:
`C-with-qmain -> HeliosObject -> SeleneObject (link HeliosQIS) -> SeleneExecutable (link libselene.so)`

In [ ]:
import tempfile
from pathlib import Path
from typing import Any

from selene_core import BuildPlanner
from selene_core.build_utils import ArtifactKind, Step, BuildCtx, Artifact
from selene_core.build_utils.utils import invoke_zig
from selene_core.build_utils.builtins import (
    register_builtins,
    HeliosObjectFileKind,
    SeleneExecutableKind,
)

from selene_core import LibDep
from selene_helios_qis_plugin import HeliosInterface
from selene_sim.build import build

## 1) Define custom artifact kind + step

In [ ]:
class CFileWithQMainKind(ArtifactKind[Path]):
    priority = 5000

    @classmethod
    def matches(cls, resource: Any) -> bool:
        if not isinstance(resource, Path):
            return False
        if not resource.is_file() or resource.suffix != ".c":
            return False
        text = resource.read_text(encoding="utf-8", errors="ignore")
        return "int qmain" in text


class CFileWithQMainToHeliosObjectStep(Step):
    input_kind = CFileWithQMainKind
    output_kind = HeliosObjectFileKind

    @classmethod
    def get_cost(cls, build_ctx: BuildCtx) -> float:
        return 10

    @classmethod
    def apply(cls, build_ctx: BuildCtx, input_artifact: Artifact) -> Artifact:
        out_path = build_ctx.artifact_dir / "program.helios.o"
        if build_ctx.verbose:
            print(f"Compiling C source to Helios object: {out_path}")
        invoke_zig(
            "cc",
            "-c",
            input_artifact.resource,
            "-o",
            out_path,
            verbose=build_ctx.verbose,
        )
        return cls._make_artifact(out_path)

## 2) Create a demo C input resource

This source intentionally references Helios entry symbols and defines `qmain`.

In [ ]:
work = Path(tempfile.mkdtemp(prefix="selene_planner_ext_"))
c_file = work / "program.c"
c_file.write_text(
    """
extern void setup(void);
extern void teardown(void);

int qmain(void) {
    setup();
    teardown();
    return 0;
}
"""
)
c_file

## 3) Register with planner and inspect planned step sequence

Here we register builtins, then add our kind + step.

In [ ]:
planner = BuildPlanner()
register_builtins(planner)
planner.add_kind(CFileWithQMainKind)
planner.add_step(CFileWithQMainToHeliosObjectStep)

interface = HeliosInterface()
interface.register_build_steps(planner)
deps = [LibDep.from_plugin(interface)]

artifact_dir = work / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
ctx = BuildCtx(artifact_dir=artifact_dir, deps=deps, cfg={}, verbose=True)

input_kind = planner.identify_kind(c_file)
steps = planner.get_optimal_steps_between(input_kind, SeleneExecutableKind, ctx)
print("input kind:", input_kind.__name__)
print("planned steps:")
for s in steps:
    print(" -", s.__name__, "::", s.input_kind.__name__, "->", s.output_kind.__name__)

## 4) Build through `selene_sim.build` with the extended planner

This executes the registered pipeline and produces a Selene executable artifact.

In [ ]:
runner = build(
    c_file,
    planner=planner,
    interface=interface,
    name="c_qmain_build_demo",
    verbose=True,
)

print("executable:", runner.executable)
print("exists:", runner.executable.exists())
print("artifacts:", list(runner.artifacts.iterdir()))

## Notes

- `build(...)` defaults to the global `DEFAULT_BUILD_PLANNER` if you do not pass `planner=`.
- For production extensions, package your kinds/steps and expose a helper like `register_my_builtins(planner)`.
- You can also register on the default planner for global behavior, but a dedicated planner is usually easier to reason about in notebooks/tests.